# EPC Ingestion — Bronze → Silver

Reads all London EPC certificate CSVs, filters to social rented, cleans, writes to silver Parquet.

In [1]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim, when, to_date, year as spark_year
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('epc_ingest') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')

Spark ready


## 1. Read all certificate CSVs into Bronze

In [2]:
RAW_PATH    = '../data/bronze/epc_raw/certificates-*.csv'
SILVER_PATH = '../data/silver/epc'

# Read all years at once — Spark globs the wildcard
raw = spark.read.csv(RAW_PATH, header=True, inferSchema=False)

print(f'Total rows across all years: {raw.count():,}')
print(f'Columns: {len(raw.columns)}')
raw.printSchema()

Total rows across all years: 23,546,857
Columns: 93
root
 |-- certificate_number: string (nullable = true)
 |-- address1: string (nullable = true)
 |-- address2: string (nullable = true)
 |-- address3: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- posttown: string (nullable = true)
 |-- address: string (nullable = true)
 |-- constituency: string (nullable = true)
 |-- constituency_label: string (nullable = true)
 |-- local_authority: string (nullable = true)
 |-- local_authority_label: string (nullable = true)
 |-- built_form: string (nullable = true)
 |-- co2_emiss_curr_per_floor_area: string (nullable = true)
 |-- co2_emissions_current: string (nullable = true)
 |-- co2_emissions_potential: string (nullable = true)
 |-- construction_age_band: string (nullable = true)
 |-- current_energy_efficiency: string (nullable = true)
 |-- current_energy_rating: string (nullable = true)
 |-- energy_consumption_current: string (nullable = true)
 |-- energy_consumption_pote

## 2. Explore tenure values — what does 'social rented' look like?

In [3]:
# Discovery step: what tenure labels exist in the raw data? This runs BEFORE any London or social-rented filter.
# Goal: identify all the label variants we need to catch (multiple spellings mean the same thing).
raw_total = raw.count()
print(f'Raw dataset (all UK, all tenures): {raw_total:,} rows')
print('Top 20 tenure labels — note the three "social" variants (rental/rented/social rented):')
print()
display(raw.groupBy('tenure').count().orderBy('count', ascending=False).limit(20).toPandas().style.format(thousands=","))

Raw dataset (all UK, all tenures): 23,546,857 rows
Top 20 tenure labels — note the three "social" variants (rental/rented/social rented):



,tenure,count
0,owner-occupied,"8,863,555"
1,rented (private),"3,788,578"
2,rented (social),"3,467,613"
3,Owner-occupied,"2,594,955"
4,unknown,"2,310,577"
5,None,"990,384"
6,Rented (social),"780,305"
7,Rented (private),"631,066"
8,Unknown,"119,819"
9,N/A,3


## 3. Filter → Deduplicate → Clean → Silver

**Why deduplicate?** The raw data contains **~1.28× more certificates than unique properties** — retrofit re-inspections, tenancy-change EPCs, and historical backlog uploads all put multiple certificates on the same property. Left uncorrected this inflates every downstream count by ~22–28% and can cause the same property to appear as both "below C" (old cert) and "at C" (post-retrofit cert), muddling the retrofit signal.

**Dedup strategy:**
1. **Primary key: UPRN** (Unique Property Reference Number) — populated on 97.6% of London social rented certificates.
2. **Fallback: `postcode|address1`** — used for the 2.4% where UPRN is missing.
3. **Rule: keep the certificate with the latest `inspection_date` per property** — this reflects current condition, which is what matters for the 2030 retrofit target.

Silver output: **one row per property**, ~479k rows (down from 612k raw), all at their most recent EPC assessment.

In [4]:
from pyspark.sql import Window
from pyspark.sql.functions import coalesce, concat_ws, desc, row_number

# ── 1. Project raw → typed silver columns (include uprn, address1 for dedup key) ──
projected = (
    raw
    .select(
        trim(col('certificate_number')).alias('certificate_id'),
        trim(col('uprn')).alias('uprn'),
        trim(col('postcode')).alias('postcode'),
        trim(col('address1')).alias('address1'),
        trim(col('local_authority')).alias('local_authority_code'),
        trim(col('local_authority_label')).alias('borough'),
        trim(col('tenure')).alias('tenure'),
        trim(col('property_type')).alias('property_type'),
        trim(col('built_form')).alias('built_form'),
        trim(col('construction_age_band')).alias('construction_age_band'),
        trim(col('current_energy_rating')).alias('epc_rating'),
        col('current_energy_efficiency').cast(FloatType()).alias('epc_score'),
        col('total_floor_area').cast(FloatType()).alias('floor_area_m2'),
        trim(col('main_fuel')).alias('main_fuel'),
        col('co2_emissions_current').cast(FloatType()).alias('co2_emissions'),
        col('energy_consumption_current').cast(FloatType()).alias('energy_consumption'),
        col('heating_cost_current').cast(FloatType()).alias('heating_cost'),
        trim(col('walls_energy_eff')).alias('walls_energy_eff'),
        trim(col('roof_energy_eff')).alias('roof_energy_eff'),
        trim(col('windows_energy_eff')).alias('windows_energy_eff'),
        trim(col('mains_gas_flag')).alias('mains_gas_flag'),
        to_date(col('inspection_date'), 'yyyy-MM-dd').alias('inspection_date'),
        trim(col('region')).alias('region_code'),
    )
    # filter: London social rented, with an EPC rating
    .filter(upper(col('tenure')).isin('RENTAL (SOCIAL)', 'SOCIAL RENTED', 'RENTED (SOCIAL)'))
    .filter(col('region_code') == 'E12000007')
    .filter(col('epc_rating').isNotNull())
)

n_pre_dedup = projected.count()
print(f'Before dedup — London social rented certs: {n_pre_dedup:,}')

# ── 2. Deduplicate: one row per property, latest inspection_date ──
# Primary key = UPRN; fallback = postcode|address1 for the 2.4% missing UPRN.
dedup_key = coalesce(
    when((col('uprn').isNotNull()) & (col('uprn') != ''), col('uprn')),
    concat_ws('|', col('postcode'), col('address1')),
)
w = Window.partitionBy(dedup_key).orderBy(desc('inspection_date'), desc('certificate_id'))
deduped = (
    projected
    .withColumn('_rn', row_number().over(w))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

# ── 3. Derive analysis columns after dedup ──
silver = (
    deduped
    .withColumn('below_epc_c', when(col('epc_rating').isin('D', 'E', 'F', 'G'), True).otherwise(False))
    .withColumn('inspection_year', spark_year(col('inspection_date')))
    .withColumn(
        'fuel_category',
        when(col('main_fuel').isin('mains gas (not community)', 'Gas: mains gas'), 'gas_individual')
        .when(col('main_fuel') == 'mains gas (community)', 'gas_community')
        .when(col('main_fuel').isin('electricity (not community)', 'Electricity: electricity, unspecified tariff'), 'electric_individual')
        .when(col('main_fuel') == 'electricity (community)', 'electric_community')
        .when(col('main_fuel').isNotNull(), 'other')
        .otherwise(None)
    )
)

n_silver = silver.count()
print(f'After dedup  — unique properties (silver):  {n_silver:,}')
print(f'Duplication removed: {n_pre_dedup - n_silver:,} certs ({(1 - n_silver/n_pre_dedup)*100:.1f}%)')
print()
display(silver.limit(5).toPandas().style.format(thousands=","))

Before dedup — London social rented certs: 612,357
After dedup  — unique properties (silver):  491,869
Duplication removed: 120,488 certs (19.7%)



,certificate_id,uprn,postcode,address1,local_authority_code,borough,tenure,property_type,built_form,construction_age_band,epc_rating,epc_score,floor_area_m2,main_fuel,co2_emissions,energy_consumption,heating_cost,walls_energy_eff,roof_energy_eff,windows_energy_eff,mains_gas_flag,inspection_date,region_code,below_epc_c,inspection_year,fuel_category
0,0158-2830-6338-9795-6631,10000001451,HA5 3HR,Flat 1,E09000015,Harrow,rented (social),Flat,Mid-Terrace,England and Wales: 1900-1929,E,51.000000,47.000000,electricity (not community),4.900000,622.000000,649.000000,Poor,N/A,Average,Y,2015-07-10,E12000007,True,"2,015",electric_individual
1,2212-6014-4166-7131-8083,10000005125,HA2 0US,3 EASTWAY CRESCENT,E09000015,Harrow,rented (social),House,Mid-Terrace,England and Wales: 1996-2002,C,75.000000,81.000000,mains gas (not community),2.200000,157.000000,395.000000,Good,Good,Average,Y,2022-08-22,E12000007,False,"2,022",gas_individual
2,2801-9119-1711-8111-0111,10000005464,HA2 0XJ,17 Union Terrace,E09000015,Harrow,rented (social),Flat,End-Terrace,England and Wales: 2003-2006,C,80.000000,40.000000,mains gas (not community),0.900000,130.000000,236.000000,Good,N/A,Good,Y,2023-11-21,E12000007,False,"2,023",gas_individual
3,2119-8112-2018-1791-0197,10000005562,HA2 0WG,"13, Karma Way, Rayners Lane",E09000015,Harrow,rented (social),House,Mid-Terrace,England and Wales: 2003-2006,C,76.000000,101.000000,mains gas (not community),2.500000,143.000000,413.000000,Good,Good,Good,Y,2022-08-22,E12000007,False,"2,022",gas_individual
4,2145-2171-8527-1812-1551,10000005565,HA2 0WG,23 Karma Way,E09000015,Harrow,rented (social),House,Semi-Detached,England and Wales: 2003-2006,C,75.000000,80.000000,mains gas (not community),2.100000,151.000000,584.000000,Good,Good,Good,Y,2023-10-04,E12000007,False,"2,023",gas_individual


## 4. Null audit

In [5]:
from pyspark.sql.functions import count, isnan

def null_count_expr(c, dtype):
    if dtype in ('double', 'float'):
        return count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    return count(when(col(c).isNull(), c)).alias(c)

display(silver.select([null_count_expr(c, t) for c, t in silver.dtypes]).toPandas().style.format(thousands=","))

,certificate_id,uprn,postcode,address1,local_authority_code,borough,tenure,property_type,built_form,construction_age_band,epc_rating,epc_score,floor_area_m2,main_fuel,co2_emissions,energy_consumption,heating_cost,walls_energy_eff,roof_energy_eff,windows_energy_eff,mains_gas_flag,inspection_date,region_code,below_epc_c,inspection_year,fuel_category
0,0,"12,949",0,1,0,242,0,0,114,0,0,0,0,"3,593","8,961",0,0,0,0,0,"92,397",0,0,0,0,"3,593"


## 5. EPC rating distribution — sanity check

In [6]:
display(silver.groupBy('epc_rating').count().orderBy('epc_rating').toPandas().style.format(thousands=","))

# What % is below C?
total = silver.count()
below_c = silver.filter(col('below_epc_c') == True).count()
print(f'Below EPC C: {below_c:,} / {total:,} = {below_c/total*100:.1f}%')

Below EPC C: 195,204 / 491,869 = 39.7%


,epc_rating,count
0,A,315
1,B,"26,820"
2,C,"269,530"
3,D,"165,099"
4,E,"26,610"
5,F,"2,731"
6,G,764


### ✓ Silver layer validation

The silver output matches the exploration in notebook 01: **612,357 London social rented certificates, 42.4% below EPC C**. The tenure-variant catchall filter (three label spellings) and the below-C flag derivation are consistent across both notebooks and match the composite figure used in notebooks 05–08.

## 6. Write to Silver Parquet, partitioned by borough

In [7]:
silver.write \
    .mode('overwrite') \
    .partitionBy('borough') \
    .parquet(SILVER_PATH)

print(f'Silver EPC written to {SILVER_PATH}')

# Verify
verify = spark.read.parquet(SILVER_PATH)
print(f'Verification read: {verify.count():,} rows')

Silver EPC written to ../data/silver/epc
Verification read: 491,869 rows
